In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard Components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output

import plotly.express as px
import pandas as pd
import base64

# CRUD Module
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################

db = AnimalShelter()

df = pd.DataFrame.from_records(db.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Logo
image_filename = 'Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
)

app.layout = html.Div([

    html.Center([

        html.Img(
            src='data:image/png;base64,{}'.format(
                encoded_image.decode()
            ),
            style={'height': '250px'}
        ),

        html.H1(
            'Grazioso Salvare Search & Rescue Dashboard'
        ),

        html.H3(
            'Charles Roaf'
        )
    ]),

    html.Hr(),

    dcc.RadioItems(
        id='filter-type',

        options=[
            {
                'label': 'Water Rescue',
                'value': 'water'
            },

            {
                'label': 'Mountain/Wilderness Rescue',
                'value': 'mountain'
            },

            {
                'label': 'Disaster/Tracking Rescue',
                'value': 'disaster'
            },

            {
                'label': 'Reset',
                'value': 'reset'
            }
        ],

        value='reset',

        labelStyle={
            'display': 'inline-block',
            'margin-right': '20px'
        }
    ),

    html.Hr(),

    dash_table.DataTable(

        id='datatable-id',

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        filter_action='native',
        sort_action='native',
        sort_mode='multi',

        page_action='native',
        page_size=10,

        row_selectable='single',
        selected_rows=[0],
        selected_columns=[],

        style_table={
            'overflowX': 'auto'
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        style={
            'display': 'flex'
        },

        children=[

            html.Div(
                id='graph-id',
                style={
                    'width': '50%'
                }
            ),

            html.Div(
                id='map-id',
                style={
                    'width': '50%'
                }
            )

        ]
    )
])

#############################################
# Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':

        query = {
            "$and": [
                {
                    "breed": {
                        "$in": [
                            "Labrador Retriever Mix",
                            "Chesapeake Bay Retriever",
                            "Newfoundland"
                        ]
                    }
                },
                {
                    "sex_upon_outcome": {
                        "$in": [
                            "Intact Female",
                            "Intact Male"
                        ]
                    }
                },
                {
                    "age_upon_outcome_in_weeks": {
                        "$gte": 26,
                        "$lte": 156
                    }
                }
            ]
        }

    elif filter_type == 'mountain':

        query = {
            "$and": [
                {
                    "breed": {
                        "$in": [
                            "German Shepherd",
                            "German Shepherd Mix",
                            "Alaskan Malamute",
                            "Old English Sheepdog",
                            "Siberian Husky",
                            "Siberian Husky Mix",
                            "Rottweiler",
                            "Rottweiler Mix"
                        ]
                    }
                },
                {
                    "sex_upon_outcome": {
                        "$in": [
                            "Intact Female",
                            "Intact Male"
                        ]
                    }
                },
                {
                    "age_upon_outcome_in_weeks": {
                        "$gte": 26,
                        "$lte": 156
                    }
                }
            ]
        }

    elif filter_type == 'disaster':

        query = {
            "$and": [
                {
                    "breed": {
                        "$in": [
                            "Doberman Pinsch",
                            "Doberman Pinsch Mix",
                            "German Shepherd",
                            "German Shepherd Mix",
                            "Golden Retriever",
                            "Bloodhound",
                            "Rottweiler",
                            "Rottweiler Mix"
                        ]
                    }
                },
                {
                    "sex_upon_outcome": {
                        "$in": [
                            "Intact Female",
                            "Intact Male"
                        ]
                    }
                },
                {
                    "age_upon_outcome_in_weeks": {
                        "$gte": 20,
                        "$lte": 300
                    }
                }
            ]
        }

    else:

        query = {}

    dff = pd.DataFrame.from_records(
        db.read(query)
    )

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')

#############################################
# Pie Chart
#############################################

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame(viewData)

    # Handle empty result sets
    if len(dff) == 0:
        return [
            html.H3("No matching records found.")
        ]

    # Get top 10 breeds
    breeds = (
        dff['breed']
        .value_counts()
        .reset_index()
        .head(10)
    )

    breeds.columns = ['breed', 'count']

    return [

        dcc.Graph(

            figure=px.pie(
                breeds,
                names='breed',
                values='count',
                title='Top 10 Breed Distribution of Rescue Candidates'
            )

        )

    ]
#############################################
# Column Highlighting
#############################################

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [
        {
            'if': {
                'column_id': col
            },
            'background_color': '#D2F3FF'
        }
        for col in selected_columns
    ]

#############################################
# Map
#############################################

@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if len(dff) == 0:
        return []

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [

        dl.Map(

            style={
                'width': '100%',
                'height': '500px'
            },

            center=[30.75, -97.48],

            zoom=10,

            children=[

                dl.TileLayer(),

                dl.Marker(

                    position=[
                        dff.iloc[row]['location_lat'],
                        dff.iloc[row]['location_long']
                    ],

                    children=[

                        dl.Tooltip(
                            str(dff.iloc[row]['breed'])
                        ),

                        dl.Popup([

                            html.H3("Animal Name"),

                            html.P(
                                str(dff.iloc[row]['name'])
                            )

                        ])

                    ]

                )

            ]

        )

    ]

# Run Dashboard
app.run_server()

Dash app running on https://recorduser-catalogjames-3000.codio.io/proxy/8050/
